In [ ]:
!pip install tensorflow numpy pillow

In [ ]:
from google.colab import files
files.upload()

Saving archive.zip to archive.zip


In [ ]:
!ls

'archive (1).zip'   archive.zip   data	 sample_data


In [ ]:
import zipfile

with zipfile.ZipFile("archive.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/data")

In [ ]:
!ls /content/data

'garbage classification'	       one-indexed-files-notrash_val.txt
'Garbage classification'	       one-indexed-files.txt
 one-indexed-files-notrash_test.txt    zero-indexed-files.txt
 one-indexed-files-notrash_train.txt


In [ ]:
!ls "/content/data/Garbage classification"

'Garbage classification'


In [ ]:
!ls "/content/data/Garbage classification/Garbage classification"

cardboard  glass  metal  paper	plastic  trash


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

In [ ]:
base_model = MobileNetV2(input_shape=(160,160,3), include_top=False, weights="imagenet")

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
output = Dense(6, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATASET_PATH = "/content/data/Garbage classification/Garbage classification"

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(160,160),
    batch_size=16,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(160,160),
    batch_size=16,
    class_mode='categorical',
    subset='validation'
)

Found 2024 images belonging to 6 classes.
Found 503 images belonging to 6 classes.


In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

Epoch 1/5
127/127 ━━━━━━━━━━━━━━━━━━━━ 244s 2s/step - accuracy: 0.6477 - loss: 1.0031 - val_accuracy: 0.3121 - val_loss: 7.6244
Epoch 2/5
127/127 ━━━━━━━━━━━━━━━━━━━━ 204s 2s/step - accuracy: 0.7717 - loss: 0.6755 - val_accuracy: 0.2863 - val_loss: 6.0301
Epoch 3/5
127/127 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.8246 - loss: 0.5269 - val_accuracy: 0.3499 - val_loss: 5.9825
Epoch 4/5
127/127 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.8547 - loss: 0.4469 - val_accuracy: 0.2008 - val_loss: 18.5007
Epoch 5/5
127/127 ━━━━━━━━━━━━━━━━━━━━ 206s 2s/step - accuracy: 0.8730 - loss: 0.3728 - val_accuracy: 0.3857 - val_loss: 3.6376


In [ ]:
model.save("model_fixed.keras")

In [ ]:
from google.colab import files
files.download("model_fixed.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from tensorflow.keras.models import load_model

# force load (ignore Keras 3 issues)
model = load_model("model_fixed.keras", compile=False, safe_mode=False)

# IMPORTANT: re-save in TF compatible format
model.save("model_final.h5")

In [ ]:
from google.colab import files
files.download("model_final.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from tensorflow.keras.models import load_model

# load old model safely
model = load_model("model_fixed.keras", compile=False, safe_mode=False)

# IMPORTANT: clean re-build
model.compile(optimizer="adam", loss="categorical_crossentropy")

# re-save clean model
model.save("model_clean.h5")

In [ ]:
from google.colab import files
files.download("model_clean.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from tensorflow.keras.models import load_model

# तुम्हारा existing model load करो
model = load_model("model_fixed.keras", compile=False, safe_mode=False)

# SAFE SAVE (TensorFlow compatible)
model.save("final_model.h5")

In [ ]:
from google.colab import files
files.download("final_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls

archive.zip  final_model.h5  model_final.h5	sample_data
data	     model_clean.h5  model_fixed.keras


In [ ]:
files.download("final_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(weights='imagenet', include_top=False)

/tmp/ipykernel_21736/3951017827.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
for layer in base_model.layers:
    layer.trainable = False

In [4]:
from tensorflow.keras import layers, models

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
output = layers.Dense(2, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)